# QM-sym C4h Symmetry Transform

This notebook implements data augmentation by applying one random operation from the C4h point group to a single QM-sym C4h molecule.

The QM-sym paper describes molecules generated and optimized under point-group constraints, with subgroup/atom labels stored in the xyz comment line. For C4h, use the eight operations generated by a fourfold rotation axis and a horizontal mirror plane:

`E, C4, C2, C4^3, i, S4^3, sigma_h, S4`.

Convention used here: the principal C4 axis is the z axis and the horizontal mirror plane is the xy plane. The local ASE pickle was produced by centering molecules in a 30 Angstrom non-periodic cubic cell, so the default symmetry center is the cell center when a cell is present.

In [ ]:
from __future__ import annotations

from dataclasses import dataclass, field
from pathlib import Path, PurePosixPath
import copy
import json
import pickle
import tarfile

from jax import config
config.update("jax_enable_x64", True)
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np

ASE_PICKLE = Path("data/qm_sym_c4h_2/qm_sym_c4h_2_ase_u0_ha.pkl")
ASE_METADATA = Path("data/qm_sym_c4h_2/qm_sym_c4h_2_ase_u0_ha.json")

assert ASE_PICKLE.exists(), ASE_PICKLE
assert ASE_METADATA.exists(), ASE_METADATA

## Load One C4h Example

Directly loading `qm_sym_c4h_2_ase_u0_ha.pkl` requires `ase`, because the pickle stores ASE `Atoms` objects. The current `DAB` environment has JAX and matplotlib but may not have `ase`. The loader below first tries the pickle path; if `ase` is unavailable, it reconstructs the same molecule from the source xyz member recorded in the adjacent JSON metadata. The transform function itself accepts either an ASE `Atoms` object or this small fallback molecule object.

In [ ]:
@dataclass
class SimpleMolecule:
    symbols: list[str]
    positions: np.ndarray
    cell: np.ndarray = field(default_factory=lambda: np.diag([30.0, 30.0, 30.0]))
    pbc: np.ndarray = field(default_factory=lambda: np.array([False, False, False]))
    info: dict = field(default_factory=dict)

    def get_positions(self) -> np.ndarray:
        return np.asarray(self.positions, dtype=np.float64).copy()

    def set_positions(self, positions) -> None:
        self.positions = np.asarray(positions, dtype=np.float64).copy()

    def get_chemical_symbols(self) -> list[str]:
        return list(self.symbols)

    def get_cell(self) -> np.ndarray:
        return np.asarray(self.cell, dtype=np.float64).copy()

    def get_pbc(self) -> np.ndarray:
        return np.asarray(self.pbc, dtype=bool).copy()

    def copy(self) -> "SimpleMolecule":
        return SimpleMolecule(
            symbols=list(self.symbols),
            positions=self.get_positions(),
            cell=self.get_cell(),
            pbc=self.get_pbc(),
            info=copy.deepcopy(self.info),
        )


def _center_positions_like_converter(positions: np.ndarray, cell_size: float = 30.0) -> np.ndarray:
    """Mirror scripts/qm_sym_to_ase.py: Atoms(..., cell=[30]*3); atoms.center()."""
    positions = np.asarray(positions, dtype=np.float64)
    midpoint = 0.5 * (positions.min(axis=0) + positions.max(axis=0))
    return positions + (0.5 * cell_size - midpoint)


def _parse_xyz_member(raw: bytes, metadata_record: dict, archive_name: str, cell_size: float = 30.0) -> SimpleMolecule:
    lines = [line.strip() for line in raw.decode("utf-8").splitlines() if line.strip()]
    num_atoms = int(lines[0])
    atom_lines = lines[2 : 2 + num_atoms]
    symbols = []
    positions = []
    mulliken_charges = []
    for atom_line in atom_lines:
        symbol, x, y, z, charge = atom_line.split()
        symbols.append(symbol)
        positions.append([float(x), float(y), float(z)])
        mulliken_charges.append(float(charge))

    positions = _center_positions_like_converter(np.asarray(positions), cell_size=cell_size)
    return SimpleMolecule(
        symbols=symbols,
        positions=positions,
        cell=np.diag([cell_size, cell_size, cell_size]),
        info={
            "source_archive": archive_name,
            "source_member": metadata_record["source_member"],
            "qm_sym_properties": metadata_record["properties"],
            "target_energy_ha": metadata_record["target_energy_ha"],
            "mulliken_charges": mulliken_charges,
            "loader": "xyz fallback reconstructed from metadata",
        },
    )


def _load_fallback_from_xyz(index: int = 0) -> tuple[SimpleMolecule, str]:
    with ASE_METADATA.open("r", encoding="utf8") as handle:
        metadata = json.load(handle)

    record = metadata["records"][index]
    archive_path = Path(metadata["source_archive"])
    target_member = record["source_member"]

    with tarfile.open(archive_path, "r") as tar:
        for member in tar.getmembers():
            if member.isfile() and PurePosixPath(member.name).name == target_member:
                handle = tar.extractfile(member)
                if handle is None:
                    raise ValueError(f"Could not extract {member.name!r}")
                return _parse_xyz_member(handle.read(), record, archive_path.name), "xyz fallback"

    raise FileNotFoundError(f"Could not find {target_member!r} in {archive_path}")


def load_qm_sym_c4h2_example(index: int = 0, prefer_pickle: bool = True):
    if prefer_pickle:
        try:
            with ASE_PICKLE.open("rb") as handle:
                images = pickle.load(handle)
            return images[index], "ASE pickle"
        except ModuleNotFoundError as exc:
            if exc.name != "ase":
                raise
            print("ASE is not installed in this kernel; using the xyz metadata fallback for the demo.")
            print("To load the pickle directly in DAB, install ASE, e.g. `conda install -n DAB -c conda-forge ase`.")

    return _load_fallback_from_xyz(index=index)


molecule, load_source = load_qm_sym_c4h2_example(index=0)
print(f"Loaded {len(molecule.get_chemical_symbols())} atoms via {load_source}.")
print("source_member:", getattr(molecule, "info", {}).get("source_member"))
print("symmetry_group:", getattr(molecule, "info", {}).get("qm_sym_properties", {}).get("symmetry_group"))

## C4h Random Transform

The pure coordinate operation is done with JAX arrays. The wrapper copies the input molecule and writes the transformed coordinates back to the copy. Atom order is intentionally preserved; the operation moves each atom to its symmetry-equivalent position in space, while downstream atomistic code can still treat the object as an unordered set of atoms.

In [ ]:
C4H_OPERATION_LABELS = ("E", "C4", "C2", "C4^3", "i", "S4^3", "sigma_h", "S4")


def c4h_operation_matrices(dtype=jnp.float64) -> jnp.ndarray:
    """Return C4h operation matrices in the same order as C4H_OPERATION_LABELS."""
    return jnp.asarray(
        [
            [[1, 0, 0], [0, 1, 0], [0, 0, 1]],      # E
            [[0, -1, 0], [1, 0, 0], [0, 0, 1]],     # C4 about z
            [[-1, 0, 0], [0, -1, 0], [0, 0, 1]],    # C2 about z
            [[0, 1, 0], [-1, 0, 0], [0, 0, 1]],     # C4^3 about z
            [[-1, 0, 0], [0, -1, 0], [0, 0, -1]],   # inversion, i = C2 * sigma_h
            [[0, 1, 0], [-1, 0, 0], [0, 0, -1]],    # S4^3
            [[1, 0, 0], [0, 1, 0], [0, 0, -1]],     # sigma_h, reflection in xy
            [[0, -1, 0], [1, 0, 0], [0, 0, -1]],    # S4
        ],
        dtype=dtype,
    )


def _positions(data) -> np.ndarray:
    if hasattr(data, "get_positions"):
        return np.asarray(data.get_positions(), dtype=np.float64)
    if isinstance(data, dict):
        return np.asarray(data["positions"], dtype=np.float64)
    return np.asarray(data.positions, dtype=np.float64)


def _cell(data) -> np.ndarray | None:
    if hasattr(data, "get_cell"):
        cell = np.asarray(data.get_cell(), dtype=np.float64)
    elif isinstance(data, dict) and "cell" in data:
        cell = np.asarray(data["cell"], dtype=np.float64)
    elif hasattr(data, "cell"):
        cell = np.asarray(data.cell, dtype=np.float64)
    else:
        return None

    if cell.shape == (3,):
        cell = np.diag(cell)
    if cell.shape != (3, 3) or np.allclose(cell, 0.0):
        return None
    return cell


def infer_symmetry_center(data) -> jnp.ndarray:
    cell = _cell(data)
    if cell is not None:
        return jnp.asarray(0.5 * cell.sum(axis=0), dtype=jnp.float64)
    return jnp.mean(jnp.asarray(_positions(data), dtype=jnp.float64), axis=0)


def _copy_with_positions(data, positions: np.ndarray):
    if hasattr(data, "copy") and hasattr(data, "set_positions"):
        out = data.copy()
        out.set_positions(positions)
        return out
    if isinstance(data, dict):
        out = copy.deepcopy(data)
        out["positions"] = np.asarray(positions, dtype=np.float64)
        return out
    out = copy.deepcopy(data)
    out.positions = np.asarray(positions, dtype=np.float64)
    return out


def _record_transform_info(data, operation_index: int, operation_label: str, matrix: np.ndarray, center: np.ndarray) -> None:
    payload = {
        "operation_index": operation_index,
        "operation_label": operation_label,
        "matrix": matrix.tolist(),
        "center": center.tolist(),
    }
    if isinstance(data, dict):
        data.setdefault("info", {})["c4h_transform"] = payload
    elif hasattr(data, "info"):
        data.info = dict(data.info)
        data.info["c4h_transform"] = payload


def c4h_random_transform(data, key):
    """Return a copy of one QM-sym C4h molecule after one random C4h operation.

    Parameters
    ----------
    data:
        One ASE Atoms object loaded from qm_sym_c4h_2_ase_u0_ha.pkl, or an object
        exposing get_positions/copy/set_positions.
    key:
        JAX PRNG key used to choose one of the eight C4h operations.
    """
    operations = c4h_operation_matrices()
    operation_index = jax.random.randint(key, shape=(), minval=0, maxval=len(C4H_OPERATION_LABELS))
    matrix = operations[operation_index]

    positions = jnp.asarray(_positions(data), dtype=jnp.float64)
    center = infer_symmetry_center(data)
    transformed_positions = center + (positions - center) @ matrix.T

    operation_index_int = int(np.asarray(jax.device_get(operation_index)))
    operation_label = C4H_OPERATION_LABELS[operation_index_int]
    transformed = _copy_with_positions(data, np.asarray(jax.device_get(transformed_positions)))
    _record_transform_info(
        transformed,
        operation_index=operation_index_int,
        operation_label=operation_label,
        matrix=np.asarray(jax.device_get(matrix)),
        center=np.asarray(jax.device_get(center)),
    )
    return transformed


demo_key = jax.random.PRNGKey(6)  # deterministic demo key; seed 6 selects C4 with current JAX.
transformed = c4h_random_transform(molecule, demo_key)
print("operation:", transformed.info["c4h_transform"]["operation_label"])
print("center:", transformed.info["c4h_transform"]["center"])

## Visualize Before and After

In [ ]:
ELEMENT_COLORS = {
    "H": "#d9d9d9",
    "B": "#e6a23c",
    "C": "#333333",
    "N": "#2f6fdb",
    "O": "#d64545",
    "F": "#38a169",
    "Cl": "#22863a",
    "Br": "#8b4513",
}

COVALENT_RADII = {
    "H": 0.31,
    "B": 0.85,
    "C": 0.76,
    "N": 0.71,
    "O": 0.66,
    "F": 0.57,
    "Cl": 1.02,
    "Br": 1.20,
}


def _symbols(data) -> list[str]:
    if hasattr(data, "get_chemical_symbols"):
        return data.get_chemical_symbols()
    if isinstance(data, dict):
        return list(data["symbols"])
    return list(data.symbols)


def _pair_limits(*molecules, pad: float = 0.75):
    coords = np.vstack([_positions(mol) for mol in molecules])
    low = coords.min(axis=0)
    high = coords.max(axis=0)
    center = 0.5 * (low + high)
    half_width = 0.5 * float(np.max(high - low)) + pad
    return [(center[i] - half_width, center[i] + half_width) for i in range(3)]


def _draw_bonds(ax, positions: np.ndarray, symbols: list[str]) -> None:
    for i in range(len(symbols)):
        for j in range(i + 1, len(symbols)):
            threshold = 1.25 * (COVALENT_RADII.get(symbols[i], 0.75) + COVALENT_RADII.get(symbols[j], 0.75))
            if np.linalg.norm(positions[i] - positions[j]) <= threshold:
                xs, ys, zs = zip(positions[i], positions[j])
                ax.plot(xs, ys, zs, color="#9aa0a6", linewidth=0.8, alpha=0.55)


def plot_molecule(ax, data, title: str, limits) -> None:
    positions = _positions(data)
    symbols = _symbols(data)
    _draw_bonds(ax, positions, symbols)
    for symbol in sorted(set(symbols)):
        mask = np.array([item == symbol for item in symbols])
        ax.scatter(
            positions[mask, 0],
            positions[mask, 1],
            positions[mask, 2],
            s=42 if symbol != "H" else 18,
            color=ELEMENT_COLORS.get(symbol, "#7f7f7f"),
            edgecolor="black",
            linewidth=0.35,
            label=symbol,
            depthshade=True,
        )
    ax.set_title(title)
    ax.set_xlabel("x / Angstrom")
    ax.set_ylabel("y / Angstrom")
    ax.set_zlabel("z / Angstrom")
    ax.set_xlim(*limits[0])
    ax.set_ylim(*limits[1])
    ax.set_zlim(*limits[2])
    ax.set_box_aspect((1, 1, 1))
    ax.legend(loc="upper right", fontsize=8, frameon=False)


limits = _pair_limits(molecule, transformed)
operation = transformed.info["c4h_transform"]["operation_label"]

fig = plt.figure(figsize=(13, 6))
ax0 = fig.add_subplot(1, 2, 1, projection="3d")
ax1 = fig.add_subplot(1, 2, 2, projection="3d")
plot_molecule(ax0, molecule, "Before", limits)
plot_molecule(ax1, transformed, f"After random C4h operation: {operation}", limits)
plt.tight_layout()
plt.show()